In [ ]:
!pip install -q onnx-graphsurgeon onnxruntime


In [ ]:
import os, json, glob
import numpy as np

CANDIDATE_DIRS = [
    "/kaggle/input/competitions/neurogolf-2026",
    "/kaggle/input/neurogolf-2026",
]
TASK_DIR = next((d for d in CANDIDATE_DIRS if os.path.isdir(d)), None)
TASK_FILES = sorted(glob.glob(os.path.join(TASK_DIR, "task*.json"))) if TASK_DIR else []
print(f"Task directory: {TASK_DIR}  (found {len(TASK_FILES)} tasks)")

def load_task(path):
    """Return (train_pairs, test_pairs)."""
    with open(path) as f:
        data = json.load(f)
    def to_grid(g):
        return np.asarray(g, dtype=np.int8) if g is not None else None
    train = [(to_grid(p["input"]), to_grid(p["output"])) for p in data.get("train", [])]
    test  = [(to_grid(p["input"]), to_grid(p.get("output"))) for p in data.get("test",  [])]
    return train, test

SUBMISSION_DIR = "/kaggle/working/submission"
os.makedirs(SUBMISSION_DIR, exist_ok=True)


In [ ]:
"""
NeuroGolf 2026 - Hierarchical Triple Sieve, corrected end-to-end.

The pipeline follows the stated three-tier plan:

  Tier 0 (Sub-Grid Normalizer / Pre-Filter)
      * grid_to_onehot / onehot_to_grid pad every ARC grid into the fixed
        (1, 10, 30, 30) float32 one-hot canvas the competition grader expects.
      * out-of-bounds cells are all-zero (no channel lit) so downstream tiers
        can distinguish real color-0 pixels from padding.

  Tier 1 (D4 pre-check)
      * analyzes the 8 dihedral symmetries of the padded canvas and, when one
        matches every training pair exactly, compiles it as
        Transpose + Slice(steps=-1).  Zero parameters, zero MACs.

  Tier 2 (Composable LUT Flow - CLF)
      * pure per-color / isomorphic transforms.  Instead of the previous
        4-plane gray-code LUT stack (5 nodes + 4 tables), we compile them
        as a single Gather on the channel axis: output = Gather(input, perm[10], axis=1).
        Cost: 10 int64 = 80 bytes total, zero MACs, bit-exact.

  Tier 3 (Implicit Graph Tsetlin Machine)
      * relational spatial logic when CLF cannot fit.  For every output color
        c we search for a small disjunction of clauses of the form
             self=a  AND  neighbor_{dir}=b
        that covers every training pixel of color c and never fires on
        pixels of other colors.  Compiled as Slice (channel extraction)
        + Slice+Concat (Zero-Parameter Topology Trick for N/S/E/W shifts)
        + Mul (AND on one-hot planes) + Max (OR across clauses) + Concat
        (assemble the 10 output channels).  No parameter-heavy adjacency
        matrix; every neighbor view is a virtual slice/concat.

  Tier 4 (Compression Pipeline)
      * every candidate is executed under onnxruntime against every training
        pair before it is emitted - a graph that does not verify is thrown
        away, never shipped.
      * golf_model strips every non-essential protobuf byte (producer names,
        doc strings, initializer names, value_info, opset domain, metadata
        props) and pins IR=10 / opset=10 to match the competition reference.

Supplemental tiers (tile, upscale, translation, constant) run alongside
the CLF and GraphTM as cheap short-circuits for common ARC transformations.
"""

import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper
import onnxruntime as ort

MAX_H = MAX_W = 30
N_COLORS = 10
GRID_SHAPE = (1, N_COLORS, MAX_H, MAX_W)
OPSET = 10
IR_VERSION = 10
_ORT_OPTS = ort.SessionOptions()
_ORT_OPTS.log_severity_level = 3


# ---------------------------------------------------------------------------
# Sub-Grid Normalizer (Tier 0)
# ---------------------------------------------------------------------------
def grid_to_onehot(grid):
    """Pad an ARC grid (H,W int) into a (1,10,30,30) float32 one-hot canvas."""
    h, w = grid.shape
    oh = np.zeros(GRID_SHAPE, dtype=np.float32)
    idx = grid.astype(np.int64)
    hh, ww = np.mgrid[0:h, 0:w]
    oh[0, idx, hh, ww] = 1.0
    return oh


def onehot_to_grid(oh, h, w):
    """Decode a padded one-hot tensor and crop back to (h, w)."""
    active = oh[0].sum(0) > 0.5
    argm = oh[0].argmax(0)
    argm = np.where(active, argm, 0)
    return argm[:h, :w].astype(np.int8)


# ---------------------------------------------------------------------------
# Compression Pipeline (Tier 4) - protobuf golfing
# ---------------------------------------------------------------------------
def _short(i):
    s = ""
    i += 1
    while i:
        i, r = divmod(i - 1, 26)
        s = chr(97 + r) + s
    return s


def golf_model(model):
    model.ir_version = IR_VERSION
    model.producer_name = ""
    model.producer_version = ""
    model.domain = ""
    model.doc_string = ""
    model.model_version = 0
    del model.metadata_props[:]
    del model.opset_import[:]
    op = model.opset_import.add()
    op.domain = ""
    op.version = OPSET
    g = model.graph
    g.name = ""
    g.doc_string = ""
    del g.value_info[:]
    name_map = {}
    used_short = set()
    counter = [0]

    def rename(old):
        if old in name_map:
            return name_map[old]
        while True:
            candidate = _short(counter[0])
            counter[0] += 1
            if candidate not in used_short:
                break
        name_map[old] = candidate
        used_short.add(candidate)
        return candidate

    # Reserve i/o first so no auto-generated short name can collide.
    used_short.update({"i", "o"})
    for t, target in ((g.input[0], "i"), (g.output[0], "o")):
        name_map[t.name] = target
        t.name = target
        t.doc_string = ""
    for init in g.initializer:
        init.doc_string = ""
        init.name = rename(init.name)
    for node in g.node:
        node.name = ""
        node.doc_string = ""
        node.domain = ""
        for i, n in enumerate(node.input):
            if n:
                node.input[i] = rename(n)
        for i, n in enumerate(node.output):
            if n:
                node.output[i] = rename(n)
    return model.SerializeToString()


# ---------------------------------------------------------------------------
# ONNX builder helpers
# ---------------------------------------------------------------------------
def _tensor(name, arr):
    return numpy_helper.from_array(np.ascontiguousarray(arr), name=name)


def _vi(name, dtype, shape):
    return helper.make_tensor_value_info(name, dtype, shape)


def build_model(nodes, initializers,
                in_shape=GRID_SHAPE, out_shape=GRID_SHAPE):
    graph = helper.make_graph(
        nodes=nodes,
        name="",
        inputs=[_vi("i", TensorProto.FLOAT, in_shape)],
        outputs=[_vi("o", TensorProto.FLOAT, out_shape)],
        initializer=initializers,
    )
    return helper.make_model(
        graph,
        opset_imports=[helper.make_opsetid("", OPSET)],
        ir_version=IR_VERSION,
    )


def verify(model_bytes, train_pairs):
    """Run the compiled graph under onnxruntime and check every training pair."""
    try:
        sess = ort.InferenceSession(model_bytes, sess_options=_ORT_OPTS,
                                    providers=["CPUExecutionProvider"])
    except Exception:
        return False
    in_name = sess.get_inputs()[0].name
    for x, y in train_pairs:
        if y is None:
            continue
        if y.shape[0] > MAX_H or y.shape[1] > MAX_W:
            return False
        oh_in = grid_to_onehot(x)
        try:
            oh_out = sess.run(None, {in_name: oh_in})[0]
        except Exception:
            return False
        pred = onehot_to_grid(oh_out, y.shape[0], y.shape[1])
        if pred.shape != y.shape or not np.array_equal(pred, y):
            return False
    return True


# ---------------------------------------------------------------------------
# Identity short-circuit
# ---------------------------------------------------------------------------
def solver_identity(train_pairs):
    if not all(x.shape == y.shape and np.array_equal(x, y) for x, y in train_pairs):
        return None
    return build_model([helper.make_node("Identity", ["i"], ["o"])], [])


# ---------------------------------------------------------------------------
# Tier 2: Composable LUT Flow  (CLF)
# ---------------------------------------------------------------------------
def _color_map(train_pairs):
    """Learn a consistent per-color substitution, or return None."""
    if not all(x.shape == y.shape for x, y in train_pairs):
        return None
    fwd = {}
    for x, y in train_pairs:
        for a, b in zip(x.ravel(), y.ravel()):
            a, b = int(a), int(b)
            if fwd.setdefault(a, b) != b:
                return None
    return fwd


def solver_clf(train_pairs):
    """CLF: pure per-color substitution -> single Gather on the channel axis.

    For every output channel c we pick an input channel s such that s -> c in
    the learned color map.  Unmapped output channels are pointed at an input
    channel that never maps anywhere, so they are guaranteed to stay empty at
    test time.
    """
    fwd = _color_map(train_pairs)
    if fwd is None:
        return None
    if all(k == v for k, v in fwd.items()):
        return None  # identity - the identity solver is cheaper
    inv = np.zeros(N_COLORS, dtype=np.int64)
    used = set()
    for c in range(N_COLORS):
        srcs = [s for s, t in fwd.items() if t == c]
        if srcs:
            inv[c] = srcs[0]
            used.add(srcs[0])
    unused_inputs = [s for s in range(N_COLORS) if s not in fwd]
    ui = iter(unused_inputs)
    for c in range(N_COLORS):
        if inv[c] == 0 and 0 not in [s for s, t in fwd.items() if t == c]:
            try:
                inv[c] = next(ui)
            except StopIteration:
                inv[c] = 0
    perm = _tensor("m", inv)
    node = helper.make_node("Gather", ["i", "m"], ["o"], axis=1)
    return build_model([node], [perm])


# ---------------------------------------------------------------------------
# D4 dihedral solver  (Tier 1 pre-check)
# ---------------------------------------------------------------------------
def _apply_thv(g, T, H, V):
    r = g.T if T else g
    if H:
        r = r[:, ::-1]
    if V:
        r = r[::-1, :]
    return np.ascontiguousarray(r)


def _slice_flip_node(in_name, out_name, axis, prefix, initializers):
    size = MAX_H if axis == 2 else MAX_W
    st = _tensor(f"{prefix}s", np.array([size - 1], dtype=np.int64))
    en = _tensor(f"{prefix}e", np.array([-size - 1], dtype=np.int64))
    ax = _tensor(f"{prefix}a", np.array([axis], dtype=np.int64))
    sp = _tensor(f"{prefix}p", np.array([-1], dtype=np.int64))
    initializers += [st, en, ax, sp]
    return helper.make_node("Slice",
                            [in_name, st.name, en.name, ax.name, sp.name],
                            [out_name])


def solver_d4(train_pairs):
    if not all(x.shape == y.shape for x, y in train_pairs):
        return None
    for T in (False, True):
        for H in (False, True):
            for V in (False, True):
                if not (T or H or V):
                    continue
                ok = True
                for x, y in train_pairs:
                    xp = np.zeros((MAX_H, MAX_W), dtype=np.int8)
                    yp = np.zeros((MAX_H, MAX_W), dtype=np.int8)
                    xp[:x.shape[0], :x.shape[1]] = x
                    yp[:y.shape[0], :y.shape[1]] = y
                    if not np.array_equal(_apply_thv(xp, T, H, V), yp):
                        ok = False
                        break
                if not ok:
                    continue
                initializers = []
                nodes = []
                cur = "i"
                step = 0
                if T:
                    nxt = f"t{step}"; step += 1
                    nodes.append(helper.make_node("Transpose", [cur], [nxt],
                                                  perm=[0, 1, 3, 2]))
                    cur = nxt
                if H:
                    nxt = f"t{step}"; step += 1
                    nodes.append(_slice_flip_node(cur, nxt, 3, f"h{step}",
                                                  initializers))
                    cur = nxt
                if V:
                    nxt = f"t{step}"; step += 1
                    nodes.append(_slice_flip_node(cur, nxt, 2, f"v{step}",
                                                  initializers))
                    cur = nxt
                nodes[-1].output[0] = "o"
                return build_model(nodes, initializers)
    return None


# ---------------------------------------------------------------------------
# Tier 3: Implicit Graph Tsetlin Machine
# ---------------------------------------------------------------------------
# Neighbor directions expressed as the SHIFT (dy, dx) that must be applied
# to the tensor so that `shifted[i, j]` equals the value of that direction's
# neighbor of `x[i, j]` in the original tensor.  For instance the north
# neighbor of (i, j) is at x[i-1, j]; shifting the whole tensor DOWN by one
# row (dy=+1) places that value at (i, j), so N = (1, 0).
_DIRECTIONS = {
    "C": (0, 0),
    "N": (1, 0),
    "S": (-1, 0),
    "E": (0, -1),
    "W": (0, 1),
}


def _shift_grid_np(x, dy, dx, fill=0):
    """Numpy equivalent of the ONNX Slice+Concat shift with zero-fill."""
    h, w = x.shape
    out = np.full_like(x, fill)
    sy0, sy1 = max(0, dy), min(h, h + dy)
    sx0, sx1 = max(0, dx), min(w, w + dx)
    gy0, gy1 = max(0, -dy), min(h, h - dy)
    gx0, gx1 = max(0, -dx), min(w, w - dx)
    out[sy0:sy1, sx0:sx1] = x[gy0:gy1, gx0:gx1]
    return out


def _learn_graphtm_clauses(train_pairs):
    """Search for a small per-color clause set.

    A clause is a triple  (self_color a, direction d, neighbor_color b)
    meaning "output pixel is c iff pixel color = a AND pixel-at-direction-d
    color = b".  We pick, for each output color c, a set of clauses that
    covers every training pixel of that color and never fires for any
    training pixel of a different color.
    """
    if not all(x.shape == y.shape for x, y in train_pairs):
        return None
    # Enumerate every possible clause once.
    dir_names = list(_DIRECTIONS.keys())
    all_clauses = [(a, d, b) for a in range(N_COLORS)
                   for d in dir_names
                   for b in range(N_COLORS)]
    # For each clause, precompute the produced-color map by iterating pairs.
    # If a clause ever fires for two different y values it is discarded.
    clause_out = {}          # (a,d,b) -> output color it uniquely fires for
    clause_covers = {}       # (a,d,b) -> total pixels covered across pairs
    invalid = set()
    for a, d, b in all_clauses:
        dy, dx = _DIRECTIONS[d]
        seen_c = None
        cover = 0
        for x, y in train_pairs:
            shifted = _shift_grid_np(x, dy, dx, fill=-1)
            mask = (x == a) & (shifted == b)
            if not mask.any():
                continue
            ys = y[mask]
            uniq = np.unique(ys)
            if uniq.size != 1:
                invalid.add((a, d, b))
                seen_c = None
                break
            if seen_c is None:
                seen_c = int(uniq[0])
            elif seen_c != int(uniq[0]):
                invalid.add((a, d, b))
                seen_c = None
                break
            cover += int(mask.sum())
        if seen_c is not None and (a, d, b) not in invalid:
            clause_out[(a, d, b)] = seen_c
            clause_covers[(a, d, b)] = cover
    if not clause_out:
        return None

    # Group by output color, prefer high-coverage clauses (usually the CENTER-
    # only ones), and greedily pick the fewest clauses that cover every pixel
    # of that color across all training pairs.
    per_color = {c: [] for c in range(N_COLORS)}
    for clause, c in clause_out.items():
        per_color[c].append(clause)
    for c in per_color:
        per_color[c].sort(key=lambda k: -clause_covers[k])

    chosen = {c: [] for c in range(N_COLORS)}
    # For each training pair build the per-color target masks we must cover.
    for c in range(N_COLORS):
        # collect targets
        targets = []
        for x, y in train_pairs:
            targets.append((x, (y == c)))
        if not any(t.any() for _, t in targets):
            continue
        # Greedy set-cover.
        covered = [np.zeros_like(t, dtype=bool) for _, t in targets]
        for clause in per_color[c]:
            a, d, b = clause
            dy, dx = _DIRECTIONS[d]
            gain = 0
            fires_list = []
            for (x, tgt), cov in zip(targets, covered):
                shifted = _shift_grid_np(x, dy, dx, fill=-1)
                fires = (x == a) & (shifted == b)
                fires_list.append(fires)
                gain += int((fires & tgt & ~cov).sum())
            if gain == 0:
                continue
            chosen[c].append(clause)
            for i, fires in enumerate(fires_list):
                covered[i] = covered[i] | fires
            if all((tgt & ~cov).sum() == 0 for (_, tgt), cov in zip(targets, covered)):
                break
        for (_, tgt), cov in zip(targets, covered):
            if (tgt & ~cov).any():
                return None  # This color cannot be represented by <=K clauses.
    # Sanity: no clause chosen for color c should fire on a pixel with y != c.
    for c, clauses in chosen.items():
        for clause in clauses:
            a, d, b = clause
            dy, dx = _DIRECTIONS[d]
            for x, y in train_pairs:
                shifted = _shift_grid_np(x, dy, dx, fill=-1)
                fires = (x == a) & (shifted == b)
                if (fires & (y != c)).any():
                    return None
    # Handle output colors that never appear in training: leave empty.
    total_clauses = sum(len(v) for v in chosen.values())
    if total_clauses == 0:
        return None
    return chosen


def _channel_slice(nodes, initializers, cache, in_name, c):
    """Extract input channel c as (1,1,30,30) tensor.  Cached across clauses."""
    key = ("plane", c)
    if key in cache:
        return cache[key]
    st_name = f"psS{c}"; en_name = f"psE{c}"; ax_name = "psA"
    if ax_name not in cache:
        ax = _tensor(ax_name, np.array([1], dtype=np.int64))
        initializers.append(ax)
        cache[ax_name] = True
    st = _tensor(st_name, np.array([c], dtype=np.int64))
    en = _tensor(en_name, np.array([c + 1], dtype=np.int64))
    initializers += [st, en]
    out_name = f"p{c}"
    nodes.append(helper.make_node("Slice",
                                  [in_name, st_name, en_name, ax_name],
                                  [out_name]))
    cache[key] = out_name
    return out_name


def _spatial_shift(nodes, initializers, cache, in_name, d):
    """Zero-Parameter Topology Trick: virtual N/S/E/W neighbor view."""
    if d == "C":
        return in_name
    key = ("shift", in_name, d)
    if key in cache:
        return cache[key]
    axis = 2 if d in ("N", "S") else 3
    size = MAX_H if axis == 2 else MAX_W
    ax_name = f"sA{axis}"
    if ax_name not in cache:
        ax = _tensor(ax_name, np.array([axis], dtype=np.int64))
        initializers.append(ax)
        cache[ax_name] = True
    # zero padding row / col shared across shifts of the same shape.
    zero_key = ("zero", axis, in_name)
    if zero_key not in cache:
        shape = [1, 1, 1, MAX_W] if axis == 2 else [1, 1, MAX_H, 1]
        z = _tensor(f"z{axis}_{in_name}", np.zeros(shape, dtype=np.float32))
        initializers.append(z)
        cache[zero_key] = z.name
    zero_name = cache[zero_key]

    # Group by physical shift-then-pad pattern:
    #   S / E: drop the first row/col, append a zero row/col at the end
    #   N / W: drop the last row/col, prepend a zero row/col at the start
    if d in ("S", "E"):
        st_name = f"shS_{d}"
        if st_name not in cache:
            st = _tensor(st_name, np.array([1], dtype=np.int64))
            initializers.append(st)
            cache[st_name] = True
        en_name = f"shE_{d}"
        if en_name not in cache:
            en = _tensor(en_name, np.array([size], dtype=np.int64))
            initializers.append(en)
            cache[en_name] = True
        sliced = f"sl_{in_name}_{d}"
        nodes.append(helper.make_node("Slice",
                                      [in_name, st_name, en_name, ax_name],
                                      [sliced]))
        out = f"sh_{in_name}_{d}"
        nodes.append(helper.make_node("Concat", [sliced, zero_name], [out],
                                      axis=axis))
    else:  # "S" or "W"
        st_name = "shS0"
        if st_name not in cache:
            st = _tensor(st_name, np.array([0], dtype=np.int64))
            initializers.append(st)
            cache[st_name] = True
        en_name = f"shE0_{d}"
        if en_name not in cache:
            en = _tensor(en_name, np.array([size - 1], dtype=np.int64))
            initializers.append(en)
            cache[en_name] = True
        sliced = f"sl_{in_name}_{d}"
        nodes.append(helper.make_node("Slice",
                                      [in_name, st_name, en_name, ax_name],
                                      [sliced]))
        out = f"sh_{in_name}_{d}"
        nodes.append(helper.make_node("Concat", [zero_name, sliced], [out],
                                      axis=axis))
    cache[key] = out
    return out


def solver_graphtm(train_pairs):
    """Implicit Graph Tsetlin Machine - relational spatial rescue tier."""
    chosen = _learn_graphtm_clauses(train_pairs)
    if chosen is None:
        return None
    # If the whole map turned out to be pure CLF (only CENTER clauses with one
    # clause per color), the CLF solver is cheaper.
    if all(all(d == "C" for _, d, _ in v) and len(v) <= 1
           for v in chosen.values()):
        return None

    nodes = []
    initializers = []
    cache = {}
    channel_outputs = []  # per output color, the (1,1,30,30) mask
    for c in range(N_COLORS):
        clauses = chosen.get(c, [])
        if not clauses:
            # Emit an all-zero channel: (input * 0) sliced to that channel.
            key = ("zero_plane",)
            if key not in cache:
                z = _tensor("z_plane",
                            np.zeros((1, 1, MAX_H, MAX_W), dtype=np.float32))
                initializers.append(z)
                cache[key] = z.name
            channel_outputs.append(cache[key])
            continue
        # Build each clause: self_plane AND neighbor_plane
        clause_out_names = []
        for i, (a, d, b) in enumerate(clauses):
            self_plane = _channel_slice(nodes, initializers, cache, "i", a)
            neigh_plane = _channel_slice(nodes, initializers, cache, "i", b)
            neigh_plane = _spatial_shift(nodes, initializers, cache,
                                         neigh_plane, d)
            out_name = f"cl_{c}_{i}"
            nodes.append(helper.make_node("Mul", [self_plane, neigh_plane],
                                          [out_name]))
            clause_out_names.append(out_name)
        # OR-reduce clauses via Max.
        if len(clause_out_names) == 1:
            mask_name = clause_out_names[0]
        else:
            mask_name = f"m_{c}"
            nodes.append(helper.make_node("Max", clause_out_names, [mask_name]))
        channel_outputs.append(mask_name)

    # Concat 10 masks along channel axis to form the output tensor.
    nodes.append(helper.make_node("Concat", channel_outputs, ["o"], axis=1))
    return build_model(nodes, initializers)


# ---------------------------------------------------------------------------
# Supplemental analytical tiers - short-circuits for other common patterns
# ---------------------------------------------------------------------------
def solver_constant(train_pairs):
    ys = [y for _, y in train_pairs if y is not None]
    if len(ys) < 2:
        return None
    if not all(y.shape == ys[0].shape and np.array_equal(y, ys[0]) for y in ys):
        return None
    oh = grid_to_onehot(ys[0])
    const = _tensor("k", oh)
    nodes = [
        helper.make_node("Constant", [], ["c"], value=const),
        helper.make_node("Identity", ["c"], ["o"]),
    ]
    return build_model(nodes, [])


def _shift(x, dy, dx):
    h, w = x.shape
    out = np.zeros_like(x)
    sy0, sy1 = max(0, dy), min(h, h + dy)
    sx0, sx1 = max(0, dx), min(w, w + dx)
    gy0, gy1 = max(0, -dy), min(h, h - dy)
    gx0, gx1 = max(0, -dx), min(w, w - dx)
    out[sy0:sy1, sx0:sx1] = x[gy0:gy1, gx0:gx1]
    return out


def _detect_translation(train_pairs):
    if not all(x.shape == y.shape for x, y in train_pairs):
        return None
    max_dy = max(x.shape[0] for x, _ in train_pairs) - 1
    max_dx = max(x.shape[1] for x, _ in train_pairs) - 1
    for dy in range(-max_dy, max_dy + 1):
        for dx in range(-max_dx, max_dx + 1):
            if dy == 0 and dx == 0:
                continue
            if all(np.array_equal(_shift(x, dy, dx), y) for x, y in train_pairs):
                return dy, dx
    return None


def solver_translation(train_pairs):
    r = _detect_translation(train_pairs)
    if r is None:
        return None
    dy, dx = r
    pads = [0, 0, max(0, dy), max(0, dx), 0, 0, max(0, -dy), max(0, -dx)]
    node_pad = helper.make_node("Pad", ["i"], ["p"],
                                mode="constant", pads=pads, value=0.0)
    off_y, off_x = max(0, -dy), max(0, -dx)
    st = _tensor("cs", np.array([off_y, off_x], dtype=np.int64))
    en = _tensor("ce", np.array([off_y + MAX_H, off_x + MAX_W], dtype=np.int64))
    ax = _tensor("ca", np.array([2, 3], dtype=np.int64))
    node_slice = helper.make_node("Slice", ["p", "cs", "ce", "ca"], ["o"])
    return build_model([node_pad, node_slice], [st, en, ax])


def _detect_tile(train_pairs):
    m = n = None
    for x, y in train_pairs:
        if y.shape[0] % x.shape[0] or y.shape[1] % x.shape[1]:
            return None
        mi = y.shape[0] // x.shape[0]
        ni = y.shape[1] // x.shape[1]
        tiled = np.tile(x, (mi, ni))
        if not np.array_equal(tiled, y):
            return None
        if m is None:
            m, n = mi, ni
        elif (m, n) != (mi, ni):
            return None
    if (m, n) == (1, 1) or m is None:
        return None
    return m, n


def solver_tile(train_pairs):
    shapes = {x.shape for x, _ in train_pairs}
    if len(shapes) != 1:
        return None
    r = _detect_tile(train_pairs)
    if r is None:
        return None
    m, n = r
    h, w = train_pairs[0][0].shape
    if h * m > MAX_H or w * n > MAX_W:
        return None
    cs = _tensor("cs", np.array([0, 0], dtype=np.int64))
    ce = _tensor("ce", np.array([h, w], dtype=np.int64))
    ax = _tensor("ca", np.array([2, 3], dtype=np.int64))
    node_crop = helper.make_node("Slice", ["i", "cs", "ce", "ca"], ["c"])
    reps = _tensor("r", np.array([1, 1, m, n], dtype=np.int64))
    node_tile = helper.make_node("Tile", ["c", "r"], ["t"])
    pads = [0, 0, 0, 0, 0, 0, MAX_H - h * m, MAX_W - w * n]
    node_pad = helper.make_node("Pad", ["t"], ["o"], mode="constant",
                                pads=pads, value=0.0)
    return build_model([node_crop, node_tile, node_pad], [cs, ce, ax, reps])


def _detect_upscale(train_pairs):
    sy = sx = None
    for x, y in train_pairs:
        if y.shape[0] % x.shape[0] or y.shape[1] % x.shape[1]:
            return None
        yi = y.shape[0] // x.shape[0]
        xi = y.shape[1] // x.shape[1]
        if yi < 1 or xi < 1:
            return None
        up = np.kron(x, np.ones((yi, xi), dtype=x.dtype))
        if not np.array_equal(up, y):
            return None
        if sy is None:
            sy, sx = yi, xi
        elif (sy, sx) != (yi, xi):
            return None
    if sy is None or (sy, sx) == (1, 1):
        return None
    return sy, sx


def solver_upscale(train_pairs):
    shapes = {x.shape for x, _ in train_pairs}
    if len(shapes) != 1:
        return None
    r = _detect_upscale(train_pairs)
    if r is None:
        return None
    sy, sx = r
    h, w = train_pairs[0][0].shape
    if h * sy > MAX_H or w * sx > MAX_W:
        return None
    cs = _tensor("cs", np.array([0, 0], dtype=np.int64))
    ce = _tensor("ce", np.array([h, w], dtype=np.int64))
    ax = _tensor("ca", np.array([2, 3], dtype=np.int64))
    node_crop = helper.make_node("Slice", ["i", "cs", "ce", "ca"], ["c"])
    scales = _tensor("s",
                     np.array([1.0, 1.0, float(sy), float(sx)], dtype=np.float32))
    node_up = helper.make_node("Resize", ["c", "s"], ["u"], mode="nearest")
    pads = [0, 0, 0, 0, 0, 0, MAX_H - h * sy, MAX_W - w * sx]
    node_pad = helper.make_node("Pad", ["u"], ["o"], mode="constant",
                                pads=pads, value=0.0)
    return build_model([node_crop, node_up, node_pad], [cs, ce, ax, scales])


# ---------------------------------------------------------------------------
# Hierarchical Triple Sieve orchestrator
# ---------------------------------------------------------------------------
#  * D4 pre-check is tried first because when it fires it produces the
#    smallest possible graph (< 200 B, zero MACs).
#  * CLF is next because per-color remaps are the single most common ARC
#    pattern and it compiles to a 40-byte Gather.
#  * Then the analytical shape-changing tiers.
#  * GraphTM is last because it costs more nodes / initializers but
#    subsumes many spatial-context ARC tasks that CLF cannot express.
#  * constant runs after every structural solver so it never masks a
#    single-pair spatial task.
SOLVERS = [
    ("identity",    solver_identity),
    ("d4",          solver_d4),
    ("clf",         solver_clf),
    ("tile",        solver_tile),
    ("upscale",     solver_upscale),
    ("translation", solver_translation),
    ("graphtm",     solver_graphtm),
    ("constant",    solver_constant),
]


def generate_submission_graph(train_pairs):
    """Return (bytes, solver_name) or (None, None) if nothing verified."""
    for name, fn in SOLVERS:
        try:
            model = fn(train_pairs)
        except Exception:
            model = None
        if model is None:
            continue
        try:
            model_bytes = golf_model(model)
        except Exception:
            continue
        if verify(model_bytes, train_pairs):
            return model_bytes, name
    return None, None


In [ ]:
import zipfile, re, time
from collections import Counter


def _smoke():
    g = np.array([[1, 1, 2, 0],
                  [3, 1, 2, 5],
                  [3, 4, 4, 5]], dtype=np.int8)
    assert generate_submission_graph([(g, g)])[1] == "identity"
    perm = {0: 0, 1: 2, 2: 1, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9}
    g2 = np.vectorize(perm.get)(g).astype(np.int8)
    _, s = generate_submission_graph([(g, g2)])
    assert s == "clf", s
    # GraphTM: replace every 1 that has a 2 directly to its east with a 7.
    # Pattern requires a neighbor predicate that CLF alone cannot express.
    gg = np.array([[1, 2, 1, 2, 3],
                   [1, 0, 1, 2, 3],
                   [4, 4, 1, 0, 3]], dtype=np.int8)
    yy = gg.copy()
    for i in range(gg.shape[0]):
        for j in range(gg.shape[1] - 1):
            if gg[i, j] == 1 and gg[i, j + 1] == 2:
                yy[i, j] = 7
    _, s = generate_submission_graph([(gg, yy)])
    assert s == "graphtm", s
    # For spatial solvers the padded canvas convention only agrees with the
    # ground truth when the grid actually fills the padded canvas.  Build a
    # 30x30 grid so verify() will accept.
    rng = np.random.default_rng(0)
    big = rng.integers(0, N_COLORS, size=(MAX_H, MAX_W), dtype=np.int64).astype(np.int8)
    _, s = generate_submission_graph([(big, np.flip(big, 1))])
    assert s == "d4", s
    _, s = generate_submission_graph([(big, np.flip(big, 0))])
    assert s == "d4", s
    _, s = generate_submission_graph([(big, np.rot90(big, 1))])
    assert s == "d4", s
    trans = np.roll(big, shift=(1, 2), axis=(0, 1))
    trans[:1] = 0; trans[:, :2] = 0
    _, s = generate_submission_graph([(big, trans)])
    assert s == "translation", s
    k = np.array([[7]], dtype=np.int8)
    _, s = generate_submission_graph([(g, k), (g[:1], k)])
    assert s == "constant", s
    print("Smoke tests passed.")


_smoke()


def solve_all(task_files, out_dir=SUBMISSION_DIR, verbose=True):
    stats = Counter()
    sizes = []
    unsolved = []
    if not task_files:
        print("No task files found; nothing to do.")
        return stats
    t0 = time.time()
    for idx, path in enumerate(task_files, start=1):
        base = os.path.basename(path)
        m = re.search(r"task(\d+)\.json$", base)
        task_id = int(m.group(1)) if m else idx
        try:
            train, _test = load_task(path)
        except Exception:
            stats["load_error"] += 1
            continue
        model_bytes, name = generate_submission_graph(train)
        if model_bytes is None:
            stats["unsolved"] += 1
            unsolved.append(task_id)
            continue
        stats[name] += 1
        sizes.append(len(model_bytes))
        out_path = os.path.join(out_dir, f"task{task_id:03d}.onnx")
        with open(out_path, "wb") as f:
            f.write(model_bytes)
    dt = time.time() - t0
    if verbose:
        solved = sum(v for k, v in stats.items() if k not in ("load_error", "unsolved"))
        print(f"Solved {solved} / {len(task_files)} tasks in {dt:.1f}s")
        for k, v in stats.most_common():
            print(f"  {k:<12} {v}")
        if sizes:
            print(f"  onnx bytes  min={min(sizes)}  mean={int(sum(sizes)/len(sizes))}  max={max(sizes)}")
        if unsolved[:10]:
            print(f"  first unsolved: {unsolved[:10]}")
    return stats


_ = solve_all(TASK_FILES)

zip_path = os.path.join(os.path.dirname(SUBMISSION_DIR), "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(SUBMISSION_DIR)):
        if fn.endswith(".onnx"):
            zf.write(os.path.join(SUBMISSION_DIR, fn), arcname=fn)
print(f"Wrote {zip_path}  ({os.path.getsize(zip_path)} bytes)")
